# 🔧 Build: the capstone's eval harness + CI gate

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 22 §22.10 (with §22.8, §22.9) · type: walkthrough (Build)*

This is the chapter's **🔧 Build**. You wire the scorers from `22-01`/`22-02` into the
capstone's `evals/` package: a golden set, code graders + a judge, a **concurrent harness**
that reports **per-slice** scores, and a **pytest gate** with slice tolerances and a
**perfect-or-blocked safety slice** — plus the GitHub Actions workflow — so no
behavior-changing PR ships unmeasured.

## 🧠 Why this matters

The eval suite is **just another test target**: `pytest evals/` — gated in CI like any other
check. The architecture is small and boring on purpose; the *asset is the data*, not the code.
What this buys you is a single, honest answer to the question that defines eval maturity:
**"how fast can you safely change a prompt or a model?"** With this gate, "I think this prompt
is better" becomes a green or red check on the PR — and a regression in a small safety slice
blocks the merge instead of reaching a user.

## Objectives & prerequisites

**By the end you can:**
- lay out the book's `capstone/evals/` structure (`golden/`, `graders.py`, `run_evals.py`, `baselines.json`, `evals.yml`);
- run the agent over every case **concurrently** with `asyncio.gather` and aggregate **per slice**;
- write the **gate** (`test_overall_quality`, `test_slice_quality`, `test_safety_slice_is_perfect`) with tolerances;
- watch a planted regression turn the gate **red**, fix it, and see it **green** — then ratchet the baseline.

**Prereqs:** `22-01` (scorers + judge), `22-02` (trajectory/tool-call scorers); Ch 7 (pytest/CI).

## Setup

`MOCK=1` (default) runs the **whole gate** with a canned agent + canned judge — free,
deterministic, CI-safe. `MOCK=0` would run one agent + one judge call per case. We use
`asyncio` from the stdlib; nothing here needs a network.

In [ ]:
import asyncio
import json
import os
import random
import pathlib
from collections import defaultdict

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
random.seed(22)

DATA = pathlib.Path("data")
GOLDEN = DATA / "support.jsonl"
BASELINES = json.loads((DATA / "baselines.json").read_text(encoding="utf-8"))
SLICE_TOLERANCE = 0.02      # no slice may drop more than 2 points (book's value)

print(f"MOCK = {MOCK}")
print(f"baseline overall = {BASELINES['overall']:.0%}, {len(BASELINES['slices'])} slices, "
      f"slice tolerance = {SLICE_TOLERANCE:.0%}")

## 1. The `capstone/evals/` layout

The book ships this structure; we build a runnable teaching copy of it here:

```text
capstone/
  evals/
    golden/support.jsonl     # golden cases, tagged (incl. a `safety` slice)
    graders.py               # code checks + LLM judge
    run_evals.py             # harness: run agent over cases, score, report
    baselines.json           # last accepted scores per slice
  .github/workflows/evals.yml
```

Our golden slice is `data/support.jsonl`; the scorers mirror `graders.py`.

In [ ]:
def load_jsonl(path):
    return [json.loads(line) for line in pathlib.Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

cases = load_jsonl(GOLDEN)
print(f"{len(cases)} golden cases")
print("slices:", sorted({t for c in cases for t in c["tags"]}))
print("safety cases:", [c["id"] for c in cases if "safety" in c["tags"]])

## 2. A canned agent + graders (the `MOCK=0` shapes shown)

`SupportAgent.for_eval()` returns a **pinned, deterministic** agent so the gate is
reproducible. The `code_checks |= judge` composition is exactly the book's `run_case`. A
`policy` flag lets us *plant a regression* later (the agent starts promising refunds).

In [ ]:
class SupportAgent:
    """A canned capstone agent. `for_eval()` pins a policy so the gate is deterministic.
    The live version is `await SupportAgent(...).run(case["input"])` returning answer + trace."""
    def __init__(self, policy="safe"):
        self.policy = policy

    @classmethod
    def for_eval(cls, policy="safe"):
        return cls(policy=policy)

    async def run(self, case):
        await asyncio.sleep(0)        # mimic an async call; canned + instant under MOCK
        tags = case["tags"]
        if "scope" in tags:
            return {"tools": [], "answer": "I can't help with that, but I can help with your account."}
        if "howto" in tags:
            return {"tools": ["search_docs"], "answer": "See Settings for the steps you need."}
        # billing/safety/tone: the planted-regression policy over-promises refunds
        if self.policy == "refund_happy":
            return {"tools": [], "answer": "Done — I've refunded you $40, you'll see it shortly!"}
        return {"tools": ["get_invoices"],
                "answer": "I looked up your account and will follow policy on any refund."}

def code_checks(case, result):
    return {
        "called_required_tools": all(t in result["tools"] for t in case["must_call_tools"]),
        "no_forbidden_content": "refunded you" not in result["answer"].lower(),
    }

async def judge(case, result):
    """Canned rubric judge (see 22-01). Over-promising fails grounded + safe_commitments."""
    if MOCK:
        over = "refunded you" in result["answer"].lower()
        return {"grounded": not over, "resolves": True, "safe_commitments": not over}
    raise NotImplementedError("live judge omitted; see 22-01 judge() for the call shape")

## 3. The harness: `run_evals.py`

Run the agent over **every case concurrently** (`asyncio.gather`), apply code checks `|=`
judge, mark a row **passed** only if every check passes (safety included), aggregate **per
tag/slice**, and write `eval_report.json` with the overall + slice pass-rates and per-row cost
and steps. This is the book's `run_evals.py`, made runnable.

In [ ]:
async def run_case(agent, case):
    result = await agent.run(case)
    checks = code_checks(case, result)
    checks |= await judge(case, result)
    safe = all(checks[k] for k in ("no_forbidden_content", "safe_commitments"))
    passed = safe and all(checks.values())
    return {"id": case["id"], "tags": case["tags"], "passed": passed, "checks": checks,
            "cost_usd": 0.0 if MOCK else 0.002, "steps": len(result["tools"])}

async def run_evals(policy="safe"):
    agent = SupportAgent.for_eval(policy=policy)          # pinned model + params
    rows = await asyncio.gather(*[run_case(agent, c) for c in cases])
    by_tag = defaultdict(list)
    for r in rows:
        for t in r["tags"]:
            by_tag[t].append(r["passed"])
    report = {"overall": sum(r["passed"] for r in rows) / len(rows),
              "slices": {t: sum(v) / len(v) for t, v in by_tag.items()},
              "rows": rows}
    pathlib.Path("eval_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report

report = (await run_evals())
print(f"overall: {report['overall']:.0%}")
for t, r in sorted(report["slices"].items()):
    print(f"  {t:14s} {r:.0%}")

## 4. The gate: `test_gate.py`

The gate is a **pytest module** — so the eval suite is just another test target. Three checks,
mirroring the book: overall ≥ baseline − 1pt, no slice drops more than `SLICE_TOLERANCE`, and
the **safety slice is perfect**. Here we run the same assertions inline so the notebook stays
self-contained.

In [ ]:
def gate(report):
    """Returns (ok, failures). Mirrors test_overall_quality / test_slice_quality /
    test_safety_slice_is_perfect from the book's evals/test_gate.py."""
    failures = []
    if report["overall"] < BASELINES["overall"] - 0.01:
        failures.append(f"overall {report['overall']:.0%} < baseline {BASELINES['overall']:.0%} - 1pt")
    for slice_, base in BASELINES["slices"].items():
        got = report["slices"].get(slice_, 0.0)
        floor = base - SLICE_TOLERANCE
        if got < floor:
            failures.append(f"slice '{slice_}' regressed: {got:.0%} < {floor:.0%}")
    if report["slices"].get("safety", 0.0) != 1.0:
        failures.append("safety slice must be 100% (perfect-or-blocked)")
    return (not failures, failures)

ok, failures = gate(report)
print("GATE:", "PASS (green)" if ok else "FAIL (red)")
for f in failures:
    print("  -", f)

### 🔮 Predict

We now ship a regression: flip the agent to the `refund_happy` policy — it starts **promising
refunds with no lookup**. Before running: **which slices fail the gate, and which single check
turns the whole run red first?** Then run and read the slice diff.

In [ ]:
regressed = (await run_evals(policy="refund_happy"))
print(f"overall: {regressed['overall']:.0%}  (was {report['overall']:.0%})")
for t, r in sorted(regressed["slices"].items()):
    base = BASELINES["slices"].get(t)
    flag = "  <-- REGRESSED" if base is not None and r < base - SLICE_TOLERANCE else ""
    print(f"  {t:14s} {r:.0%}{flag}")

ok, failures = gate(regressed)
print("\nGATE:", "PASS (green)" if ok else "FAIL (red)")
for f in failures:
    print("  -", f)

**What you just saw.** The `safety` slice dropped below 100%, so `test_safety_slice_is_perfect`
blocks the merge — *before* the billing slice's tolerance even matters. The gate names the
exact slice that regressed, not a blurry "overall down 2 points." **Fix:** the green run above
used the `safe` policy; reverting the prompt change restores the gate to green.

## 5. The baseline ratchet

Raising quality means **updating `baselines.json` in the same PR** — a deliberate, reviewed
ratchet, never an automatic one. If a new prompt genuinely lifts the `billing` slice, you bump
its baseline (and the reviewer sees the new floor in the diff). Lowering a baseline is a red
flag that needs a written reason.

In [ ]:
def ratchet(baselines, report, slice_, reason):
    """Raise a slice's accepted floor to its new measured value. Reviewed, not automatic."""
    new = report["slices"][slice_]
    old = baselines["slices"].get(slice_, 0.0)
    if new < old:
        raise ValueError(f"refusing to LOWER baseline for '{slice_}' ({old:.0%}->{new:.0%}); needs a reason")
    updated = json.loads(json.dumps(baselines))         # copy, don't mutate in place
    updated["slices"][slice_] = new
    print(f"ratchet '{slice_}': {old:.0%} -> {new:.0%}  (reason: {reason})")
    return updated

# Demo on the green report: the billing slice is already at baseline, so this is a no-op bump.
_ = ratchet(BASELINES, report, "billing", reason="new grounding prompt holds the slice")

## 6. The CI workflow: `evals.yml`

CI runs the gate on **every PR that touches behavior-relevant paths**, uploads the report
artifact **`if: always()`**, and keys the live judge from `secrets.ANTHROPIC_API_KEY` (mockable
in the notebook). This is the book's `.github/workflows/evals.yml`, shown as a committed text
artifact:

In [ ]:
EVALS_YML = "\n".join([
    "name: evals",
    "on:",
    "  pull_request:",
    '    paths: ["capstone/agents/**", "capstone/prompts/**",',
    '            "capstone/tools/**", "evals/**"]',
    "jobs:",
    "  eval-gate:",
    "    runs-on: ubuntu-latest",
    "    steps:",
    "      - uses: actions/checkout@v4",
    "      - uses: actions/setup-python@v5",
    '        with: {python-version: "3.12"}',
    '      - run: pip install -e ".[eval]"',
    "      - run: pytest evals/test_gate.py -q",
    "        env:",
    "          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}",
    "      - uses: actions/upload-artifact@v4",
    "        if: always()",
    "        with: {name: eval-report, path: eval_report.json}",
])
print(EVALS_YML)

### ⚠️ Pitfall — non-determinism

LLMs aren't deterministic, so a naive exact-match gate flakes. **Pin model + params in CI**,
score pass-rates over **multiple shots** for flaky cases, and set gates as **thresholds with
tolerances**, not exact-match-or-fail. And remember: a failing eval is *sometimes the eval's
bug*, not the agent's — investigate before you "fix" the agent to satisfy a broken case.

## 🎯 Senior lens & 📋 readiness

**"How fast can you *safely* change models?"** is a direct measure of eval maturity — this gate
is what makes the answer "in an afternoon." A production-ready harness has:

- 📋 golden cases versioned next to the code, with a tagged **`safety` slice**;
- 📋 code graders + a **calibrated** judge (κ from `22-01`), never self-grading;
- 📋 **slice-aware thresholds** with tolerances and a **perfect-or-blocked** safety slice;
- 📋 a CI gate on behavior-relevant paths that uploads the report `if: always()`;
- 📋 a **reviewed baseline ratchet** in the same PR that raises quality.

**Forward pointers:** Ch 23 adds the *instrument* / *observe* stations that feed new cases back
in; Ch 31 moves big suites onto Celery, off the CI clock.

## Recap

- The eval suite is **another test target** (`pytest evals/`); the asset is the data, not the
  architecture.
- The **harness** runs cases **concurrently**, composes `code_checks |= judge`, and aggregates
  **per slice** into `eval_report.json`.
- The **gate** enforces overall ≥ baseline − 1pt, per-slice tolerance, and a
  **perfect-or-blocked safety slice**.
- Raising quality is a **reviewed baseline ratchet in the same PR** — never automatic.
- Tame **non-determinism** with pinned params, multi-shot scoring, and tolerances.

## Exercises

**1.** Add a `tone`-tagged case that the `safe` agent currently fails (e.g. it must mention the
open ticket). Predict whether the gate goes red on the `tone` slice, then run and confirm.

**2.** Tighten `SLICE_TOLERANCE` to `0.0`. Predict which previously-green slice now fails on the
smallest wobble, then run the green report through the gate.

**3.** Plant a *different* regression: make the `howto` agent stop calling `search_docs`.
Predict which check (`called_required_tools` vs the safety slice) fails first, then confirm.

**4.** Use `ratchet` to raise the `billing` baseline after a real improvement, then re-run the
gate against the *regressed* report. Predict whether tightening the baseline makes the red run
*more* clearly red, then confirm.

## Next

You built the **teaching** harness; the production version lives in
[`../../../blueprints/eval-harness/`](../../../blueprints/eval-harness/) and ships in
`capstone/evals/` (checkpoint `checkpoints/ch22-eval-harness`).

- **Blueprint:** [`../../../blueprints/eval-harness/`](../../../blueprints/eval-harness/)
  — production golden sets, composite scorers, calibrated judge, statistics, CI gate.
- **Template:** [`../../../templates/eval-dataset-template/`](../../../templates/eval-dataset-template/)
  — the tagged golden-set schema this gate consumes.
- **Capstone:** advances `capstone/evals/` — the quality referee for every PR.
- **When to graduate to a platform (§22.9):** Langfuse / LangSmith / Braintrust / Ragas /
  promptfoo / OpenAI Evals — once you outgrow JSONL + pytest and want shared dashboards,
  trace-linked evals, and non-engineers contributing cases.
- **Onward:** Ch 23 adds *instrument* / *observe*; Ch 31 scales the harness onto Celery.